In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sustfl/output2.w64
/kaggle/input/nilm-sust-v4/best_hybrid_model.pth
/kaggle/input/nilm-sust-v4/__results__.html
/kaggle/input/nilm-sust-v4/nilm_dataset_combined.csv
/kaggle/input/nilm-sust-v4/pretrained_model.pth
/kaggle/input/nilm-sust-v4/__notebook__.ipynb
/kaggle/input/nilm-sust-v4/__output__.json
/kaggle/input/nilm-sust-v4/custom.css


In [3]:
import soundfile as sf
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime, timedelta
import gc

# Try to import GPU libraries
try:
    import cudf
    import cupy as cp
    GPU_AVAILABLE = True
    print("GPU libraries loaded successfully!")
except ImportError:
    GPU_AVAILABLE = False
    print("GPU libraries not available, using CPU mode")

# Configuration
VOLTAGE_CALIBRATION = 460.0
CURRENT_CALIBRATION = 52.5

MAINS_FILE = "/kaggle/input/sustfl/output2.w64"
APPLIANCES_FOLDER = "/kaggle/input/sustdataed2/appliances"
LABELS_FOLDER = "/kaggle/input/sustdataed2/appliances/labels"
OUTPUT_FILE_PART1 = "nilm_dataset_part1.csv"
OUTPUT_FILE_PART2 = "nilm_dataset_part2.csv"


def process_mains_data(mains_file, start_timestamp=None, use_gpu=False):
    """Process mains data with optional GPU acceleration"""
    print("Processing Mains Data" + (" (GPU)" if use_gpu else " (CPU)"))
    print("=" * 50)
    
    # Read audio file
    data, samplerate = sf.read(mains_file)
    print(f"Sample Rate: {samplerate} Hz")
    print(f"Original Data Shape: {data.shape}")
    
    if use_gpu and GPU_AVAILABLE:
        # GPU processing
        data_gpu = cp.asarray(data, dtype=cp.float32)
        
        voltage = cp.asnumpy(data_gpu[:, 2] * VOLTAGE_CALIBRATION)
        current = cp.asnumpy(data_gpu[:, 3] * CURRENT_CALIBRATION)
        active_power = cp.asnumpy(data_gpu[:, 0] * VOLTAGE_CALIBRATION * CURRENT_CALIBRATION)
        reactive_power = cp.asnumpy(data_gpu[:, 1] * VOLTAGE_CALIBRATION * CURRENT_CALIBRATION)
        
        del data_gpu
        cp.get_default_memory_pool().free_all_blocks()
    else:
        # CPU processing
        voltage = data[:, 2] * VOLTAGE_CALIBRATION
        current = data[:, 3] * CURRENT_CALIBRATION
        active_power = data[:, 0] * VOLTAGE_CALIBRATION * CURRENT_CALIBRATION
        reactive_power = data[:, 1] * VOLTAGE_CALIBRATION * CURRENT_CALIBRATION
    
    # Create DataFrame
    df_calibrated = pd.DataFrame({
        'Voltage_V': voltage.astype('float32'),
        'Current_A': current.astype('float32'),
        'Active_Power_W': active_power.astype('float32'),
        'Reactive_Power_VAR': reactive_power.astype('float32')
    })
    
    del data, voltage, current, active_power, reactive_power
    gc.collect()
    
    # Create timestamps
    if start_timestamp is None:
        start_timestamp = datetime(2016, 10, 24, 0, 0, 0)
    
    print("Creating timestamps...")
    time_delta = timedelta(seconds=1/samplerate)
    timestamps = [start_timestamp + i * time_delta for i in range(len(df_calibrated))]
    
    df_calibrated['timestamp'] = timestamps
    df_calibrated = df_calibrated.set_index('timestamp')
    
    print(f"Calibrated Data Shape: {df_calibrated.shape}")
    print(f"Date range: {df_calibrated.index.min()} to {df_calibrated.index.max()}")
    
    return df_calibrated


def load_appliance_data(appliances_folder, labels_folder, mains_start_time, mains_end_time):
    """Load appliance data"""
    print("\nLoading Appliance Data")
    print("=" * 50)
    
    appliances = {}
    
    # Load appliance data
    appliance_files = glob.glob(os.path.join(appliances_folder, "*.csv"))
    print(f"Found {len(appliance_files)} appliance files")
    
    for file_path in appliance_files:
        filename = os.path.basename(file_path)
        print(f"Loading: {filename}", end=" ")
        
        try:
            # Read with pandas
            data = pd.read_csv(file_path)
            
            if 'timestamp' not in data.columns:
                print("- No timestamp column")
                continue
            
            # Convert timestamp
            data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
            data = data.dropna(subset=['timestamp'])
            data = data.set_index('timestamp')
            
            # Filter by mains time range
            data_filtered = data[(data.index >= mains_start_time) & (data.index <= mains_end_time)]
            
            if len(data_filtered) > 0:
                appliance_name = filename.split('.')[0].split('_', 1)[1] if '_' in filename else filename.split('.')[0]
                appliances[appliance_name] = data_filtered
                print(f"- Loaded {len(data_filtered):,} samples")
            else:
                print("- No data in time range")
                
        except Exception as e:
            print(f"- Error: {e}")
    
    return appliances


def create_unified_dataset_status_only(mains_data, appliances):
    """Create unified dataset with only Status columns (no Power columns)"""
    print("\nCreating Unified Dataset with Status Columns Only")
    print("=" * 50)
    
    # Start with calibrated mains data
    unified_df = mains_data.copy()
    print(f"Base dataset shape: {unified_df.shape}")
    
    # Power thresholds for ON/OFF detection
    power_thresholds = {
        'CoffeeMachine': 10, 'Fridge-Freezer': 5, 'Freezer': 5, 'HandMixer': 10,
        'HairDryer-Straightener': 20, 'Kettle': 50, 'MacBook2007': 5,
        'MacBookPro2011-1': 5, 'MacBookPro2011-2': 5, 'Microwave': 20,
        'Stove-Oven': 50, 'TV-Philips': 5, 'TV-Sharp': 5, 'TV-Grundig': 5,
        'TV-Samsung': 5, 'TV-LG': 5, 'Toaster': 50, 'VaccumCleaner': 100
    }
    
    if len(appliances) == 0:
        print("WARNING: No appliances loaded!")
        unified_df['Number_Active_Devices'] = 0
        return unified_df
    
    # Add appliance status data only (no power data)
    print("\nAdding appliance status data:")
    status_columns = []
    
    for appliance_name, app_data in appliances.items():
        status_col_name = f"{appliance_name}_Status"
        
        # Find power column
        power_col = None
        for col in ['power', 'Power', 'P', 'consumption']:
            if col in app_data.columns:
                power_col = col
                break
        
        if power_col:
            # Create temporary dataframe with status
            temp_df = app_data[[power_col]].copy()
            
            # Calculate status based on power threshold
            threshold = power_thresholds.get(appliance_name, 10)
            temp_df[status_col_name] = (temp_df[power_col] > threshold).astype(np.int8)
            
            # Merge only the status column with unified dataset
            unified_df = pd.merge(
                unified_df, 
                temp_df[[status_col_name]], 
                left_index=True, 
                right_index=True, 
                how='left'
            )
            
            # Fill NaN with 0 (device is OFF)
            unified_df[status_col_name] = unified_df[status_col_name].fillna(0).astype(np.int8)
            status_columns.append(status_col_name)
            
            # Count active samples
            active_samples = unified_df[status_col_name].sum()
            print(f"  {appliance_name}: {active_samples:,} active samples ({active_samples/len(unified_df)*100:.2f}%)")

    # Calculate number of active devices
    print("\nCalculating number of active devices...")
    if status_columns:
        unified_df['Number_Active_Devices'] = unified_df[status_columns].sum(axis=1).astype(np.int8)
        
        # Display statistics
        avg_active = unified_df['Number_Active_Devices'].mean()
        max_active = unified_df['Number_Active_Devices'].max()
        print(f"Average active devices: {avg_active:.2f}")
        print(f"Maximum active devices: {max_active}")
    else:
        unified_df['Number_Active_Devices'] = 0
        print("No status columns found - setting Number_Active_Devices to 0")
    
    print(f"\nFinal dataset shape: {unified_df.shape}")
    print(f"Total columns: {len(unified_df.columns)}")
    
    return unified_df


def split_and_save_dataset(unified_df, output_file1, output_file2):
    """Split dataset into two equal parts and save"""
    print("\nSplitting and Saving Dataset")
    print("=" * 50)
    
    total_rows = len(unified_df)
    split_point = total_rows // 2
    
    print(f"Total rows: {total_rows:,}")
    print(f"Split point: {split_point:,}")
    
    print(f"\nSaving Part 1 to: {output_file1}")
    part1 = unified_df.iloc[:split_point]
    part1.to_csv(output_file1)
    print(f"Part 1 saved: {len(part1):,} rows")
    print(f"Date range: {part1.index.min()} to {part1.index.max()}")
    
    del part1
    gc.collect()
    
    # Part 2: Second half
    print(f"\nSaving Part 2 to: {output_file2}")
    part2 = unified_df.iloc[split_point:]
    part2.to_csv(output_file2)
    print(f"Part 2 saved: {len(part2):,} rows")
    print(f"Date range: {part2.index.min()} to {part2.index.max()}")
    
    del part2
    gc.collect()
    
    print("\n Both parts saved successfully!")


def main():
    print("NILM Dataset Processing - Status Columns Only")
    print("=" * 70)
    
    # Check GPU
    use_gpu = GPU_AVAILABLE
    if use_gpu:
        print(f"GPU Mode: Enabled")
        try:
            print(f"GPU Device: {cp.cuda.Device().name}")
        except:
            pass
    else:
        print("GPU Mode: Disabled (using CPU)")
    print("=" * 70)
    
    # Step 1: Process mains data
    mains_data = process_mains_data(MAINS_FILE, use_gpu=use_gpu)
    
    # Step 2: Load appliance data
    appliances = load_appliance_data(
        APPLIANCES_FOLDER, 
        LABELS_FOLDER, 
        mains_data.index.min(), 
        mains_data.index.max()
    )
    
    # Step 3: Create unified dataset with status columns only
    unified_df = create_unified_dataset_status_only(mains_data, appliances)
    
    # Step 4: Split and save dataset
    split_and_save_dataset(unified_df, OUTPUT_FILE_PART1, OUTPUT_FILE_PART2)
    
    # Display final information
    print("\n" + "="*70)
    print("DATASET SUMMARY")
    print("="*70)
    print(f"Total samples: {len(unified_df):,}")
    print(f"Total columns: {len(unified_df.columns)}")
    print(f"Date range: {unified_df.index.min()} to {unified_df.index.max()}")
    
    # Memory usage
    memory_mb = unified_df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory Usage: {memory_mb:.2f} MB")
    
    # Show column structure
    print("\n" + "="*70)
    print("COLUMN STRUCTURE")
    print("="*70)
    print(" Mains Features (4):")
    print("  - Voltage_V, Current_A, Active_Power_W, Reactive_Power_VAR")
    
    status_cols = [col for col in unified_df.columns if col.endswith('_Status')]
    print(f"\n✓ Appliance Status Columns ({len(status_cols)}):")
    for col in status_cols:
        active_percent = unified_df[col].sum() / len(unified_df) * 100
        print(f"  - {col} ({active_percent:.2f}% active)")
    
    print(f"\n✓ Aggregated Feature (1):")
    print("  - Number_Active_Devices (int8)")
    
    # Statistics
    print("\n" + "="*70)
    print("STATISTICS")
    print("="*70)
    print(f"Average active devices: {unified_df['Number_Active_Devices'].mean():.2f}")
    print(f"Max active devices: {unified_df['Number_Active_Devices'].max()}")
    
    # Device activity summary
    print(f"\nDevice Activity Summary:")
    for col in status_cols:
        active_count = unified_df[col].sum()
        active_percent = active_count / len(unified_df) * 100
        print(f"  {col}: {active_count:,} samples ({active_percent:.2f}%)")
    
    print("\n" + "="*70)
    print("OUTPUT FILES")
    print("="*70)
    print(f" {OUTPUT_FILE_PART1} - First half of dataset")
    print(f" {OUTPUT_FILE_PART2} - Second half of dataset")
    
    print("\n" + "="*70)
    print(" Processing Complete!")
    print("="*70)
    
    # Clean up
    if GPU_AVAILABLE:
        cp.get_default_memory_pool().free_all_blocks()
    gc.collect()


if __name__ == "__main__":
    main()

GPU libraries loaded successfully!
NILM Dataset Processing - Status Columns Only
GPU Mode: Enabled
Processing Mains Data (GPU)
Sample Rate: 50 Hz
Original Data Shape: (40151016, 4)
Creating timestamps...
Calibrated Data Shape: (40151016, 4)
Date range: 2016-10-24 00:00:00 to 2016-11-02 07:03:40.300000

Loading Appliance Data
Found 18 appliance files
Loading: 14_TV-Grundig.csv - No data in time range
Loading: 16_TV-LG.csv - Loaded 388,840 samples
Loading: 17_Toaster.csv - Loaded 388,866 samples
Loading: 4_HandMixer.csv - Loaded 388,857 samples
Loading: 9_MacBookPro2011-2.csv - Loaded 388,902 samples
Loading: 8_MacBookPro2011-1.csv - Loaded 388,880 samples
Loading: 18_VaccumCleaner.csv - Loaded 388,867 samples
Loading: 11_Stove-Oven.csv - Loaded 388,857 samples
Loading: 12_TV-Philips.csv - Loaded 388,890 samples
Loading: 2_Fridge-Freezer.csv - Loaded 388,868 samples
Loading: 1_CoffeeMachine.csv - Loaded 388,859 samples
Loading: 10_Microwave.csv - Error: Invalid comparison between dtype=d

In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================================
# Part 1: MobileNetV3 Components
# ============================================================================

class HardSwish(nn.Module):
    """Hard Swish: x * ReLU6(x + 3) / 6"""
    def forward(self, x):
        return x * F.relu6(x + 3.0) / 6.0

class SqueezeExcitation(nn.Module):
    """SE Block with proper implementation"""
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, max(in_channels // reduction, 8)),  # Minimum 8
            nn.ReLU(inplace=True),
            nn.Linear(max(in_channels // reduction, 8), in_channels),
            nn.Hardsigmoid()
        )
    
    def forward(self, x):
        b, c, _ = x.size()
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y

class InvertedResidual(nn.Module):
    """MobileNetV3 Inverted Residual Block"""
    def __init__(self, in_ch, out_ch, kernel_size, stride, expand_ratio, use_se=True, use_hs=True):
        super().__init__()
        hidden_dim = int(round(in_ch * expand_ratio))
        self.use_residual = stride == 1 and in_ch == out_ch
        
        layers = []
        # Expand
        if expand_ratio != 1:
            layers.extend([
                nn.Conv1d(in_ch, hidden_dim, 1, bias=False),
                nn.BatchNorm1d(hidden_dim),
                HardSwish() if use_hs else nn.ReLU(inplace=True)
            ])
        
        # Depthwise
        layers.extend([
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size, stride, 
                     padding=kernel_size//2, groups=hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            HardSwish() if use_hs else nn.ReLU(inplace=True)
        ])
        
        # SE
        if use_se:
            layers.append(SqueezeExcitation(hidden_dim))
        
        # Project
        layers.extend([
            nn.Conv1d(hidden_dim, out_ch, 1, bias=False),
            nn.BatchNorm1d(out_ch)
        ])
        
        self.conv = nn.Sequential(*layers)
    
    def forward(self, x):
        if self.use_residual:
            return x + self.conv(x)
        return self.conv(x)

class MobileNetV3Encoder(nn.Module):
    """MobileNetV3-Large backbone for feature extraction"""
    def __init__(self, input_channels=4):
        super().__init__()
        
        # Stem
        self.stem = nn.Sequential(
            nn.Conv1d(input_channels, 16, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm1d(16),
            HardSwish()
        )
        
        # MobileNetV3 blocks (simplified but powerful)
        # [in, out, kernel, stride, expand, se, hs]
        configs = [
            [16, 16, 3, 1, 1, False, False],
            [16, 24, 3, 2, 4, False, False],
            [24, 24, 3, 1, 3, False, False],
            [24, 40, 5, 2, 3, True, False],
            [40, 40, 5, 1, 3, True, False],
            [40, 80, 3, 2, 6, False, True],
            [80, 80, 3, 1, 2.5, False, True],
            [80, 112, 3, 1, 6, True, True],
            [112, 160, 5, 1, 6, True, True],
        ]
        
        layers = []
        for in_ch, out_ch, k, s, e, se, hs in configs:
            layers.append(InvertedResidual(in_ch, out_ch, k, s, e, se, hs))
        self.blocks = nn.Sequential(*layers)
        
        # Head
        self.conv_head = nn.Sequential(
            nn.Conv1d(160, 960, 1, bias=False),
            nn.BatchNorm1d(960),
            HardSwish()
        )
        
        self.pool = nn.AdaptiveAvgPool1d(1)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.conv_head(x)
        x = self.pool(x)
        return x.flatten(1)  # (batch, 960)

# ============================================================================
# Part 2: VAE Components
# ============================================================================

class VAEEncoder(nn.Module):
    """VAE Encoder for learning latent representations"""
    def __init__(self, input_dim=400, hidden_dim=256, latent_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)
    
    def forward(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

class VAEDecoder(nn.Module):
    """VAE Decoder for reconstruction"""
    def __init__(self, latent_dim=128, hidden_dim=256, output_dim=400):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, z):
        return self.decoder(z)

# ============================================================================
# Part 3: Hybrid Model (MobileNetV3 + VAE)
# ============================================================================

class HybridNILM(nn.Module):
    """
    Hybrid Architecture:
    - MobileNetV3: Strong supervised classification
    - VAE: Anomaly detection & new appliance discovery
    """
    def __init__(self, input_channels=4, n_appliances=9, embedding_dim=128, 
                 vae_latent_dim=64, use_vae=True, window_size=100):
        super().__init__()
        self.n_appliances = n_appliances
        self.use_vae = use_vae
        self.embedding_dim = embedding_dim
        
        # MobileNetV3 Backbone
        self.mobilenet = MobileNetV3Encoder(input_channels)
        
        # Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(960, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            HardSwish(),
            nn.Dropout(0.3),
            nn.Linear(embedding_dim, n_appliances)
        )
        
        # VAE Components (optional but recommended)
        if use_vae:
            # Input: flattened signal (4 channels × window_size)
            input_dim=input_channels * window_size
            self.vae_encoder = VAEEncoder(input_dim=input_dim, latent_dim=vae_latent_dim)
            self.vae_decoder = VAEDecoder(latent_dim=vae_latent_dim, output_dim=input_dim)
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def reparameterize(self, mu, logvar):
        """VAE reparameterization trick"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x, return_all=False):
        """
        Args:
            x: (batch, 4, window_size) - raw signals
            return_all: if True, return all intermediate outputs
        
        Returns:
            classification_logits: (batch, n_appliances)
            If return_all=True: dict with all outputs
        """
        # MobileNetV3 path
        features = self.mobilenet(x)  # (batch, 960)
        logits = self.classifier(features)  # (batch, n_appliances)
        
        if not self.use_vae:
            return logits
        
        # VAE path (for anomaly detection)
        x_flat = x.view(x.size(0), -1)  # (batch, 400)
        mu, logvar = self.vae_encoder(x_flat)
        z = self.reparameterize(mu, logvar)
        x_recon = self.vae_decoder(z)
        
        if return_all:
            return {
                'logits': logits,
                'features': features,
                'mu': mu,
                'logvar': logvar,
                'z': z,
                'reconstruction': x_recon
            }
        
        return logits, x_recon, mu, logvar
    
    def compute_vae_loss(self, x_recon, x_orig, mu, logvar, beta=0.1):
        """VAE loss: Reconstruction + KL divergence"""
        x_flat = x_orig.view(x_orig.size(0), -1)
        recon_loss = F.mse_loss(x_recon, x_flat, reduction='mean')
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x_orig.size(0)
        return recon_loss + beta * kl_loss, recon_loss, kl_loss
    
    def detect_anomaly(self, x):
        """Compute anomaly score for new appliance detection"""
        self.eval()
        with torch.no_grad():
            if self.use_vae:
                _, x_recon, _, _ = self.forward(x)
                x_flat = x.view(x.size(0), -1)
                anomaly_scores = F.mse_loss(x_recon, x_flat, reduction='none').mean(dim=1)
            else:
                # Fallback: use prediction confidence
                logits = self.forward(x)
                probs = torch.sigmoid(logits)
                anomaly_scores = 1.0 - probs.max(dim=1)[0]
        return anomaly_scores

# ============================================================================
# Part 4: Training System
# ============================================================================

class HybridTrainer:
    """Complete training system for Hybrid NILM"""
    def __init__(self, csv_path, appliance_names, window_size=100, stride=50, use_vae=True):
        self.csv_path = csv_path
        self.appliance_names = appliance_names
        self.window_size = window_size
        self.stride = stride
        self.use_vae = use_vae
        self.device = device
        
        # Initialize model
        self.model = HybridNILM(
            input_channels=4,
            n_appliances=len(appliance_names),
            embedding_dim=256,
            vae_latent_dim=128,
            use_vae=use_vae,
            window_size=self.window_size
        ).to(device)
        
        print(f"{'='*70}")
        print(f" Hybrid NILM Model Initialized")
        print(f"{'='*70}")
        print(f"Architecture: MobileNetV3 + {'VAE' if use_vae else 'No VAE'}")
        print(f"Appliances: {len(appliance_names)}")
        print(f"Window Size: {window_size}, Stride: {stride}")
        print(f"Device: {device}")
        print(f"{'='*70}")
        
        self.scaler = StandardScaler()
        self.test_data = None
    
    def load_and_prepare_data(self, max_samples=8899288): 
        """Load CSV and create windows"""
        print("\n Loading data...")
        df = pd.read_csv(self.csv_path, index_col=0, parse_dates=True, nrows=max_samples)
        print(f"   Loaded {len(df)} samples")
        
        mains_cols = ['Voltage_V', 'Current_A', 'Active_Power_W', 'Reactive_Power_VAR']
        status_cols = [f"{name}_Status" for name in self.appliance_names]
        
        # Check which appliances exist
        existing_appliances = [col for col in status_cols if col in df.columns]
        print(f"   Found {len(existing_appliances)}/{len(status_cols)} appliances in data")
        
        X_windows = []
        y_labels = []
        
        print("\n Creating windows...")
        for i in range(0, len(df) - self.window_size, self.stride):
            window = df.iloc[i:i+self.window_size]
            
            # Extract features (4, window_size)
            signals = window[mains_cols].values.T
            
            # Better normalization: robust scaling per window
            signals_norm = np.zeros_like(signals)
            for ch in range(4):
                ch_data = signals[ch]
                q25, q75 = np.percentile(ch_data, [25, 75])
                iqr = q75 - q25
                if iqr > 1e-6:
                    signals_norm[ch] = (ch_data - np.median(ch_data)) / iqr
                else:
                    signals_norm[ch] = ch_data - np.median(ch_data)
            
            X_windows.append(signals_norm)
            
            # Extract labels
            labels = []
            for col in status_cols:
                if col in df.columns:
                    labels.append(int(window[col].max() > 0))
                else:
                    labels.append(0)
            y_labels.append(labels)
        
        X = np.array(X_windows, dtype=np.float32)
        y = np.array(y_labels, dtype=np.float32)
        
        print(f"   Created {len(X)} windows")
        print(f"   Shape: X={X.shape}, y={y.shape}")
        
        # Filter windows with activity
        active_mask = y.sum(axis=1) > 0
        idle_mask = y.sum(axis=1) == 0
        
        n_active = active_mask.sum()
        n_idle = idle_mask.sum()
        
        print(f"\n Data distribution:")
        print(f"   Active windows: {n_active} ({n_active/len(y)*100:.1f}%)")
        print(f"   Idle windows: {n_idle} ({n_idle/len(y)*100:.1f}%)")
        
        # Better balancing: keep more idle samples
        target_idle = int(n_active * 0.5)  # 50% idle instead of 30%
        idle_indices = np.where(idle_mask)[0]
        if len(idle_indices) > target_idle:
            selected_idle = np.random.choice(idle_indices, target_idle, replace=False)
            keep_mask = active_mask.copy()
            keep_mask[selected_idle] = True
        else:
            keep_mask = np.ones(len(y), dtype=bool)
        
        X_balanced = X[keep_mask]
        y_balanced = y[keep_mask]
        
        print(f"   Balanced dataset: {len(X_balanced)} windows")
        
        return X_balanced, y_balanced
    
    def _get_appliance_combinations(self, y):
        """Convert binary labels to combination names"""
        combinations = []
        for i in range(len(y)):
            active_appliances = []
            for j, appliance in enumerate(self.appliance_names):
                if y[i, j] == 1:
                    active_appliances.append(appliance)
            
            if active_appliances:
                combination_name = '+'.join(active_appliances)
            else:
                combination_name = 'No_Appliance'
            combinations.append(combination_name)
        
        return combinations
    
    def _filter_rare_combinations(self, X, y, min_samples=2):
        """Filter out combinations with too few samples"""
        combinations = self._get_appliance_combinations(y)
        combo_counts = Counter(combinations)
        
        # Identify combinations to keep
        valid_combinations = {combo for combo, count in combo_counts.items() if count >= min_samples}
        
        # Filter data
        keep_indices = [i for i, combo in enumerate(combinations) if combo in valid_combinations]
        
        X_filtered = X[keep_indices]
        y_filtered = y[keep_indices]
        
        print(f"   Filtered dataset: {len(X_filtered)} windows (removed {len(X) - len(X_filtered)} rare combinations)")
        print(f"   Remaining combinations: {len(valid_combinations)}")
        
        return X_filtered, y_filtered
    
    def train(self, epochs=100, batch_size=128, lr=5e-4, weight_decay=1e-4, 
              vae_weight=0.05, test_size=0.2):
        """Main training loop"""
        print(f"\n{'='*70}")
        print(" Starting Training")
        print(f"{'='*70}")
        
        # Load data
        X, y = self.load_and_prepare_data()
        
        # Filter rare combinations
        X, y = self._filter_rare_combinations(X, y, min_samples=2)
        
        # Get combination names for evaluation
        self.combination_names = self._get_appliance_combinations(y)
        
        # Split without stratification to avoid the error
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )
        
        print(f"\n Split:")
        print(f"   Train: {len(X_train)} samples")
        print(f"   Test:  {len(X_test)} samples")
        
        # Store test data
        self.test_data = {'X': X_test, 'y': y_test}
        self.test_combinations = self._get_appliance_combinations(y_test)
        
        # Create dataloaders
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train).to(device),
            torch.FloatTensor(y_train).to(device)
        )
        test_dataset = TensorDataset(
            torch.FloatTensor(X_test).to(device),
            torch.FloatTensor(y_test).to(device)
        )
        
        # Weighted sampling for class imbalance
        class_counts = y_train.sum(axis=0)
        weights = 1.0 / (class_counts + 1)
        sample_weights = (y_train * weights).sum(axis=1)
        sample_weights = np.maximum(sample_weights, 0.1)
        
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        
        # Optimizer & scheduler
        optimizer = optim.AdamW(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=5, verbose=True
        )
        
        # Loss functions
        pos_weight = torch.FloatTensor(
            (len(y_train) - y_train.sum(axis=0)) / (y_train.sum(axis=0) + 1e-8)
        ).to(device)
        cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        
        best_f1 = 0
        patience_counter = 0
        patience = 50  
        
        print(f"\n{'='*70}")
        print("Training Loop")
        print(f"{'='*70}")
        
        for epoch in range(epochs):
            # Training phase
            self.model.train()
            train_cls_loss = 0
            train_vae_loss = 0
            
            for batch_x, batch_y in train_loader:
                optimizer.zero_grad()
                
                if self.use_vae:
                    logits, x_recon, mu, logvar = self.model(batch_x)
                    
                    # Classification loss
                    cls_loss = cls_criterion(logits, batch_y)
                    
                    # VAE loss
                    vae_loss, recon_loss, kl_loss = self.model.compute_vae_loss(
                        x_recon, batch_x, mu, logvar, beta=0.1
                    )
                    
                    # Combined loss
                    loss = cls_loss + vae_weight * vae_loss
                    
                    train_vae_loss += vae_loss.item()
                else:
                    logits = self.model(batch_x)
                    loss = cls_criterion(logits, batch_y)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                
                train_cls_loss += cls_loss.item() if self.use_vae else loss.item()
            
            # Evaluation phase
            metrics = self._evaluate(test_loader)
            current_f1 = metrics['macro_f1']
            
            scheduler.step(current_f1)
            
            # Logging
            if (epoch + 1) % 5 == 0:
                log = f"Epoch [{epoch+1:3d}/{epochs}] "
                log += f"CLS: {train_cls_loss/len(train_loader):.4f} "
                if self.use_vae:
                    log += f"VAE: {train_vae_loss/len(train_loader):.4f} "
                log += f"F1: {current_f1:.4f}"
                print(log)
            
            # Save best model
            if current_f1 > best_f1:
                best_f1 = current_f1
                patience_counter = 0
                torch.save({
                    'model_state_dict': self.model.state_dict(),
                    'appliance_names': self.appliance_names,
                    'architecture': 'HybridNILM',
                    'use_vae': self.use_vae,
                    'f1_score': best_f1
                }, 'best_hybrid_model.pth')
            else:
                patience_counter += 1
            
            if patience_counter >= patience:
                print(f"\n  Early stopping at epoch {epoch+1}")
                break
        
        # Load best model
        checkpoint = torch.load('best_hybrid_model.pth', weights_only=False)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        
        print(f"\n{'='*70}")
        print(f" Training Completed!")
        print(f"{'='*70}")
        print(f"Best F1 Score: {best_f1:.4f}")
        print(f"Model saved: best_hybrid_model.pth")
        
        return self.model
    
    def _evaluate(self, test_loader):
        """Evaluate model"""
        self.model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                if self.use_vae:
                    logits, _, _, _ = self.model(batch_x)
                else:
                    logits = self.model(batch_x)
                
                preds = torch.sigmoid(logits).cpu().numpy()
                all_preds.append(preds)
                all_labels.append(batch_y.cpu().numpy())
        
        all_preds = np.vstack(all_preds)
        all_labels = np.vstack(all_labels)
        y_pred_binary = (all_preds > 0.5).astype(int)
        
        # Calculate F1 per appliance
        f1_scores = []
        for i in range(len(self.appliance_names)):
            if all_labels[:, i].sum() > 0:
                f1 = f1_score(all_labels[:, i], y_pred_binary[:, i], zero_division=0)
                f1_scores.append(f1)
        
        return {'macro_f1': np.mean(f1_scores) if f1_scores else 0}
    
    def comprehensive_evaluation(self):
        """Detailed evaluation report for appliance combinations"""
        if self.test_data is None:
            print("No test data available")
            return
        
        print(f"\n{'='*70}")
        print(" COMPREHENSIVE EVALUATION - APPLIANCE COMBINATIONS")
        print(f"{'='*70}")
        
        X_test = torch.FloatTensor(self.test_data['X']).to(device)
        y_test = self.test_data['y']
        
        self.model.eval()
        with torch.no_grad():
            if self.use_vae:
                logits, _, _, _ = self.model(X_test)
            else:
                logits = self.model(X_test)
            
            y_pred_probs = torch.sigmoid(logits).cpu().numpy()
            y_pred = (y_pred_probs > 0.5).astype(int)
        
        # Convert predictions to combination format
        pred_combinations = self._get_appliance_combinations(y_pred)
        true_combinations = self.test_combinations
        
        # Get unique combinations
        unique_combinations = sorted(set(true_combinations))
        
        # Generate detailed classification report
        print(f"\n{'='*70}")
        print("Detailed Classification Report for Appliance Combinations")
        print(f"{'='*70}")
        
        # Create target names for the report (only combinations with enough samples)
        target_names = []
        y_true_labels = []
        y_pred_labels = []
        
        for i, combo in enumerate(unique_combinations):
            # Only include combinations with at least 2 samples in test set
            combo_count = true_combinations.count(combo)
            if combo_count >= 2:
                target_names.append(combo)
                # Create binary labels for this combination
                y_true_combo = [1 if true_combinations[j] == combo else 0 for j in range(len(true_combinations))]
                y_pred_combo = [1 if pred_combinations[j] == combo else 0 for j in range(len(pred_combinations))]
                
                # Calculate metrics
                precision = precision_score(y_true_combo, y_pred_combo, zero_division=0)
                recall = recall_score(y_true_combo, y_pred_combo, zero_division=0)
                f1 = f1_score(y_true_combo, y_pred_combo, zero_division=0)
                
                print(f"{combo:<80} {precision:.3f}    {recall:.3f}    {f1:.3f}    {combo_count}")
        
        # Additional detailed analysis for top combinations
        print(f"\n{'='*70}")
        print("Detailed Analysis for Top Combinations")
        print(f"{'='*70}")
        
        # Count combination frequencies
        combo_counts = Counter(true_combinations)
        
        # Analyze top combinations by frequency (minimum 2 samples)
        top_combinations = [(combo, count) for combo, count in combo_counts.most_common() if count >= 2][:20]
        
        print(f"\n{'Combination':<80} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Support':<10}")
        print("-" * 120)
        
        for combo, true_count in top_combinations:
            # Calculate metrics for this combination
            y_true_combo = [1 if true_combinations[i] == combo else 0 for i in range(len(true_combinations))]
            y_pred_combo = [1 if pred_combinations[i] == combo else 0 for i in range(len(pred_combinations))]
            
            precision = precision_score(y_true_combo, y_pred_combo, zero_division=0)
            recall = recall_score(y_true_combo, y_pred_combo, zero_division=0)
            f1 = f1_score(y_true_combo, y_pred_combo, zero_division=0)
            
            print(f"{combo:<80} {precision:<10.3f} {recall:<10.3f} {f1:<10.3f} {true_count:<10}")
        
        return y_pred, y_pred_probs

Using device: cuda


In [ ]:
"""
====================================================================
FEDERATED LEARNING WITH HYBRID NILM MODEL
====================================================================
Complete implementation using your actual MobileNetV3 + VAE model
"""

# ============================================================================
# CELL 1: Import Libraries and Define Your Model Architecture
# ============================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split
from collections import Counter, defaultdict
from copy import deepcopy
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
appliance_names = [
    'CoffeeMachine', 'Fridge-Freezer', 'Freezer', 'HandMixer',
    'HairDryer-Straightener', 'Kettle', 'MacBook2007',
    'MacBookPro2011-1', 'MacBookPro2011-2', 'Microwave',
    'Stove-Oven', 'TV-Philips', 'TV-Sharp', 'TV-Grundig',
    'TV-Samsung', 'TV-LG', 'Toaster', 'VaccumCleaner'
]

window_size = 1000

# Create new model with your architecture
global_model = HybridNILM(
    input_channels=4,
    n_appliances=len(appliance_names),
    embedding_dim=128,
    vae_latent_dim=64,
    use_vae=True,
    window_size=window_size
).to(device)

n_appliances = len(appliance_names)
print(f"\nModel Configuration:")
print(f"    Number of appliances: {n_appliances}")
print(f"    Window size: {window_size}")
print(f"    Use VAE: {global_model.use_vae}")
print(f"    Model architecture: MobileNetV3 + VAE")

In [ ]:
class FederatedNILMServer:
    """Central Server for Federated Learning"""
    def __init__(self, global_model, device='cuda'):
        self.global_model = global_model
        self.device = device
        self.round_history = []
        
    def aggregate_models(self, client_models, client_weights=None):
        """FedAvg Aggregation"""
        if client_weights is None:
            client_weights = [1.0 / len(client_models)] * len(client_models)
        else:
            total = sum(client_weights)
            client_weights = [w / total for w in client_weights]
        
        global_dict = self.global_model.state_dict()
        
        for key in global_dict.keys():
            if global_dict[key].dtype == torch.long:
                continue
            global_dict[key] = torch.zeros_like(global_dict[key])
            for client_model, weight in zip(client_models, client_weights):
                global_dict[key] += weight * client_model[key].to(self.device)
        
        self.global_model.load_state_dict(global_dict)
        print(f"  Aggregated {len(client_models)} client models")
        
    def get_global_model(self):
        return deepcopy(self.global_model.state_dict())

class FederatedNILMClient:
    """Independent Client"""
    def __init__(self, client_id, model_template, appliance_names, device='cuda'):
        self.client_id = client_id
        self.device = device
        self.appliance_names = appliance_names
        self.model = deepcopy(model_template).to(device)
        self.local_data = None
        
    def load_local_data(self, X_local, y_local):
        """Load local data"""
        self.local_data = {
            'X': torch.FloatTensor(X_local).to(self.device),
            'y': torch.FloatTensor(y_local).to(self.device)
        }
        
        # Print statistics
        active_counts = y_local.sum(axis=0)
        n_active = (y_local.sum(axis=1) > 0).sum()
        
        print(f"\n  {self.client_id}:")
        print(f"     Total samples: {len(X_local)}")
        print(f"     Active windows: {n_active} ({n_active/len(y_local)*100:.1f}%)")
        
        # Show top appliances
        top_appliances = []
        for i, count in enumerate(active_counts):
            if count > 0:
                top_appliances.append((self.appliance_names[i], int(count)))
        
        top_appliances.sort(key=lambda x: x[1], reverse=True)
        if top_appliances:
            print(f"     Top appliances: {', '.join([f'{name}({cnt})' for name, cnt in top_appliances[:3]])}")
    
    def update_model(self, global_state_dict):
        """Update from server"""
        self.model.load_state_dict(global_state_dict)
    
    def local_train(self, epochs=5, batch_size=128, lr=1e-4, use_vae=True, vae_weight=0.05):
        """Local training with your model"""
        if self.local_data is None:
            raise ValueError("No local data loaded!")
        
        self.model.train()
        optimizer = optim.Adam(self.model.parameters(), lr=lr)
        
        dataset = TensorDataset(self.local_data['X'], self.local_data['y'])
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        
        # Loss function with class weights
        y_train = self.local_data['y'].cpu().numpy()
        pos_weight = torch.FloatTensor(
            (len(y_train) - y_train.sum(axis=0)) / (y_train.sum(axis=0) + 1e-8)
        ).to(self.device)
        cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        
        print(f"\n  {self.client_id} - Training:")
        
        for epoch in range(epochs):
            epoch_cls_loss = 0
            epoch_vae_loss = 0
            
            for batch_x, batch_y in loader:
                optimizer.zero_grad()
                
                if use_vae:
                    logits, x_recon, mu, logvar = self.model(batch_x)
                    
                    # Classification loss
                    cls_loss = cls_criterion(logits, batch_y)
                    
                    # VAE loss
                    vae_loss, _, _ = self.model.compute_vae_loss(
                        x_recon, batch_x, mu, logvar, beta=0.1
                    )
                    
                    # Combined loss
                    loss = cls_loss + vae_weight * vae_loss
                    
                    epoch_vae_loss += vae_loss.item()
                else:
                    logits = self.model(batch_x)
                    loss = cls_criterion(logits, batch_y)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                
                epoch_cls_loss += loss.item()
            
            avg_cls_loss = epoch_cls_loss / len(loader)
            avg_vae_loss = epoch_vae_loss / len(loader) if use_vae else 0
            
            if (epoch + 1) % 2 == 0 or epoch == epochs - 1:
                if use_vae:
                    print(f"     Epoch [{epoch+1}/{epochs}] CLS Loss: {avg_cls_loss:.4f}, VAE Loss: {avg_vae_loss:.4f}")
                else:
                    print(f"     Epoch [{epoch+1}/{epochs}] Loss: {avg_cls_loss:.4f}")
        
        # Evaluate
        metrics = self._evaluate_local(loader, use_vae)
        
        return (
            self.model.state_dict(),
            len(self.local_data['X']),
            metrics
        )
    
    def _evaluate_local(self, loader, use_vae):
        """Local evaluation"""
        self.model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch_x, batch_y in loader:
                if use_vae:
                    logits, _, _, _ = self.model(batch_x)
                else:
                    logits = self.model(batch_x)
                
                preds = torch.sigmoid(logits).cpu().numpy()
                all_preds.append(preds)
                all_labels.append(batch_y.cpu().numpy())
        
        all_preds = np.vstack(all_preds)
        all_labels = np.vstack(all_labels)
        y_pred = (all_preds > 0.5).astype(int)
        
        # F1 per appliance
        f1_scores = []
        for i in range(len(self.appliance_names)):
            if all_labels[:, i].sum() > 0:
                f1 = f1_score(all_labels[:, i], y_pred[:, i], zero_division=0)
                f1_scores.append(f1)
        
        macro_f1 = np.mean(f1_scores) if f1_scores else 0
        print(f"     Local F1: {macro_f1:.4f}")
        
        return {'macro_f1': macro_f1}

In [ ]:
class FederatedLearningCoordinator:
    """Main Coordinator"""
    def __init__(self, global_model, appliance_names, device='cuda'):
        self.server = FederatedNILMServer(global_model, device)
        self.appliance_names = appliance_names
        self.device = device
        self.clients = {}
        
    def add_client(self, client_id, X_local, y_local):
        """Add new client"""
        client = FederatedNILMClient(
            client_id=client_id,
            model_template=self.server.global_model,
            appliance_names=self.appliance_names,
            device=self.device
        )
        client.load_local_data(X_local, y_local)
        self.clients[client_id] = client
        
    def federated_training(self, n_rounds=10, clients_per_round=None, 
                          local_epochs=5, batch_size=128, lr=1e-4, use_vae=True):
        """Main FL Loop"""
        print(f"\n{'='*70}")
        print(f"FEDERATED LEARNING WITH YOUR MODEL")
        print(f"{'='*70}")
        print(f"Total Clients: {len(self.clients)}")
        print(f"Rounds: {n_rounds}")
        print(f"Local Epochs: {local_epochs}")
        print(f"Model: MobileNetV3 + VAE")
        print(f"{'='*70}")
        
        if clients_per_round is None:
            clients_per_round = len(self.clients)
        
        for round_idx in range(n_rounds):
            print(f"\n{'='*70}")
            print(f"Round [{round_idx + 1}/{n_rounds}]")
            print(f"{'='*70}")
            
            # Select clients
            selected_clients = list(self.clients.keys())
            if clients_per_round < len(self.clients):
                selected_clients = np.random.choice(
                    selected_clients,
                    size=clients_per_round,
                    replace=False
                )
            
            print(f"Selected: {', '.join(selected_clients)}")
            
            # Distribute global model
            global_weights = self.server.get_global_model()
            
            # Collect updates
            client_updates = []
            client_sizes = []
            round_metrics = []
            
            for client_id in selected_clients:
                client = self.clients[client_id]
                client.update_model(global_weights)
                
                local_weights, n_samples, metrics = client.local_train(
                    epochs=local_epochs,
                    batch_size=batch_size,
                    lr=lr,
                    use_vae=use_vae,
                    vae_weight=0.05
                )
                
                client_updates.append(local_weights)
                client_sizes.append(n_samples)
                round_metrics.append(metrics['macro_f1'])
            
            # Aggregate
            self.server.aggregate_models(client_updates, client_sizes)
            
            # Summary
            avg_f1 = np.mean(round_metrics)
            print(f"\n  Round Summary:")
            print(f"     Average F1: {avg_f1:.4f}")
            print(f"     Best Client F1: {max(round_metrics):.4f}")
            print(f"  Round {round_idx + 1} completed")
        
        print(f"\n{'='*70}")
        print("Federated Learning Completed!")
        print(f"{'='*70}")
        
        return self.server.global_model

In [ ]:
def load_complete_dataset(csv_path, appliance_names, window_size=1000, stride=100, max_rows=None):
    """
    Load complete dataset without sampling
    """
    print(f"\n{'='*70}")
    print("LOADING COMPLETE DATASET")
    print(f"{'='*70}")
    
    try:
        if max_rows:
            df = pd.read_csv(csv_path, nrows=max_rows)
        else:
            df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"Error: File not found at {csv_path}")
        return None, None
    
    print(f"Loaded complete dataset: {len(df)} rows")
    
    mains_cols = ['Voltage_V', 'Current_A', 'Active_Power_W', 'Reactive_Power_VAR']
    status_cols = [f"{name}_Status" for name in appliance_names]
    
    # Check available appliances
    available_appliances = []
    for i, appliance in enumerate(appliance_names):
        if f"{appliance}_Status" in df.columns:
            available_appliances.append(appliance)
    
    print(f"\nAvailable appliances in dataset: {len(available_appliances)}/{len(appliance_names)}")
    
    # Create windows
    X_windows = []
    y_labels = []
    
    print(f"\nCreating windows (size={window_size}, stride={stride})...")
    
    n_windows = max(1, (len(df) - window_size) // stride)
    print(f"Possible windows: {n_windows}")
    
    for i in range(0, n_windows * stride, stride):
        window = df.iloc[i:i+window_size]
        
        if len(window) < window_size:
            continue
        
        # Extract signals
        signals = window[mains_cols].values.T
        
        # Normalization per window
        signals_norm = np.zeros_like(signals)
        for ch in range(4):
            ch_data = signals[ch]
            q25, q75 = np.percentile(ch_data, [25, 75])
            iqr = q75 - q25
            if iqr > 1e-6:
                signals_norm[ch] = (ch_data - np.median(ch_data)) / iqr
            else:
                signals_norm[ch] = ch_data - np.median(ch_data)
        
        X_windows.append(signals_norm)
        
        # Extract labels
        labels = []
        for col in status_cols:
            if col in df.columns:
                labels.append(int(window[col].max() > 0))
            else:
                labels.append(0)
        y_labels.append(labels)
    
    X = np.array(X_windows, dtype=np.float32)
    y = np.array(y_labels, dtype=np.float32)
    
    print(f"\nCreated {len(X)} windows")
    print(f"Shape: X={X.shape}, y={y.shape}")
    
    # Statistics
    active_samples = (y.sum(axis=1) > 0).sum()
    idle_samples = len(y) - active_samples
    
    print(f"\nDataset Statistics:")
    print(f"    Active samples: {active_samples} ({active_samples/len(y)*100:.1f}%)")
    print(f"    Idle samples: {idle_samples} ({idle_samples/len(y)*100:.1f}%)")
    
    # Appliance occurrence
    print(f"\nAppliance Occurrence:")
    for i, appliance in enumerate(appliance_names):
        count = y[:, i].sum()
        if count > 0:
            percentage = count / len(y) * 100
            print(f"    {appliance:<25}: {count:>6} ({percentage:>5.1f}%)")
    
    return X, y

In [ ]:
def analyze_appliance_combinations(y, appliance_names, min_support=1):
    """
    Analyze combinations of appliances
    """
    print(f"\n{'='*70}")
    print("APPLIANCE COMBINATIONS ANALYSIS")
    print(f"{'='*70}")
    
    combinations_counter = Counter()
    
    for sample in y:
        active_indices = np.where(sample == 1)[0]
        if len(active_indices) == 0:
            combination_str = "No_Appliance"
        else:
            active_appliances = [appliance_names[i] for i in active_indices]
            combination_str = "+".join(sorted(active_appliances))
        
        combinations_counter[combination_str] += 1
    
    sorted_combinations = combinations_counter.most_common()
    
    print(f"\nTotal unique combinations: {len(sorted_combinations)}")
    print(f"\nTOP COMBINATIONS (Support ≥ {min_support})")
    print("-" * 100)
    print(f"{'Combination':<80} {'Support':<12}")
    print("-" * 100)
    
    for combination, support in sorted_combinations[:50]:  # Show top 50
        if support >= min_support:
            print(f"{combination:<80} {support:<12}")
    
    return combinations_counter

In [ ]:
def split_data_for_clients(X, y, n_clients=2, split_ratio=None):
    """
    Split complete dataset among clients
    """
    print(f"\n{'='*70}")
    print(f"SPLITTING DATA AMONG {n_clients} CLIENTS")
    print(f"{'='*70}")
    
    if split_ratio is None:
        split_ratio = [1.0/n_clients] * n_clients
    
    clients_data = {}
    total_samples = len(X)
    
    # Calculate split points
    split_points = []
    current_point = 0
    for i in range(n_clients):
        if i == n_clients - 1:
            split_points.append((current_point, total_samples))
        else:
            next_point = current_point + int(total_samples * split_ratio[i])
            split_points.append((current_point, next_point))
            current_point = next_point
    
    # Split data
    for i, (start_idx, end_idx) in enumerate(split_points):
        client_id = f"Client_{i+1}"
        
        clients_data[client_id] = {
            'X': X[start_idx:end_idx],
            'y': y[start_idx:end_idx]
        }
        
        n_samples = end_idx - start_idx
        n_active = (clients_data[client_id]['y'].sum(axis=1) > 0).sum()
        
        print(f"\n{client_id}:")
        print(f"    Samples: {n_samples} ({n_samples/total_samples*100:.1f}%)")
        print(f"    Active: {n_active} ({n_active/n_samples*100:.1f}%)")
        
        # Top appliances
        active_counts = clients_data[client_id]['y'].sum(axis=0)
        top_indices = np.argsort(-active_counts)[:3]
        for idx in top_indices:
            if active_counts[idx] > 0:
                print(f"    {appliance_names[idx]}: {int(active_counts[idx])}")
    
    return clients_data

In [ ]:
csv_path = '/kaggle/working/nilm_dataset_part1.csv'

# Load complete dataset
X, y = load_complete_dataset(
    csv_path=csv_path,
    appliance_names=appliance_names,
    window_size=window_size,
    stride=100,
    max_rows=2000000  
)

if X is None:
    print("Failed to load dataset. Exiting.")
    exit()

# Analyze combinations
combinations = analyze_appliance_combinations(y, appliance_names, min_support=1)

# Split among clients (60% Client 1, 40% Client 2)
N_CLIENTS = 2
CLIENT_SPLIT_RATIO = [0.6, 0.4]

clients_data = split_data_for_clients(
    X, y,
    n_clients=N_CLIENTS,
    split_ratio=CLIENT_SPLIT_RATIO
)


coordinator = FederatedLearningCoordinator(
    global_model=global_model,  # Your pretrained model
    appliance_names=appliance_names,
    device=device
)

# Add clients
for client_id, data in clients_data.items():
    coordinator.add_client(client_id, data['X'], data['y'])

print(f"\nSystem initialized with {len(coordinator.clients)} clients")

In [ ]:
print(f"\n{'='*70}")
print("RUNNING FEDERATED LEARNING")
print(f"{'='*70}")

# Training parameters
FL_ROUNDS = 12
LOCAL_EPOCHS = 4
BATCH_SIZE = 64
LEARNING_RATE = 2e-4
USE_VAE = True  
CLIENTS_PER_ROUND = N_CLIENTS  # Use both clients each round

# Start federated training
federated_model = coordinator.federated_training(
    n_rounds=FL_ROUNDS,
    clients_per_round=CLIENTS_PER_ROUND,
    local_epochs=LOCAL_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    use_vae=USE_VAE
)


print(f"\n{'='*70}")
print("SAVING FEDERATED MODEL")
print(f"{'='*70}")

# Save the federated model
torch.save({
    'model_state_dict': federated_model.state_dict(),
    'appliance_names': appliance_names,
    'architecture': 'HybridNILM_MobileNetV3_VAE',
    'window_size': window_size,
    'use_vae': USE_VAE,
    'federated_config': {
        'n_clients': N_CLIENTS,
        'n_rounds': FL_ROUNDS,
        'local_epochs': LOCAL_EPOCHS,
        'client_split_ratio': CLIENT_SPLIT_RATIO,
        'based_on_pretrained': True
    }
}, 'federated_hybrid_model.pth')

print("Federated model saved to 'federated_hybrid_model.pth'")

In [ ]:
def evaluate_combination_performance(model, X_test, y_test, appliance_names, batch_size=64, use_vae=True):
    """
    Comprehensive evaluation including combination analysis
    """
    print(f"\n{'='*70}")
    print("COMPREHENSIVE EVALUATION")
    print(f"{'='*70}")
    
    model.eval()
    
    # Create test loader
    test_dataset = TensorDataset(
        torch.FloatTensor(X_test).to(device),
        torch.FloatTensor(y_test).to(device)
    )
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    all_pred_probs = []
    all_labels = []
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            if use_vae:
                logits, _, _, _ = model(batch_x)
            else:
                logits = model(batch_x)
            
            y_pred_probs = torch.sigmoid(logits).cpu().numpy()
            all_pred_probs.append(y_pred_probs)
            all_labels.append(batch_y.cpu().numpy())
    
    y_pred_probs = np.vstack(all_pred_probs)
    y_test_all = np.vstack(all_labels)
    y_pred = (y_pred_probs > 0.5).astype(int)
    
    # 1. Per-appliance metrics
    print("\n1. PER-APPLIANCE PERFORMANCE:")
    print("-" * 80)
    print(f"{'Appliance':<25} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Support':<10}")
    print("-" * 80)
    
    appliance_metrics = {}
    f1_scores = []
    
    for i, appliance in enumerate(appliance_names):
        if y_test_all[:, i].sum() > 0:
            precision = precision_score(y_test_all[:, i], y_pred[:, i], zero_division=0)
            recall = recall_score(y_test_all[:, i], y_pred[:, i], zero_division=0)
            f1 = f1_score(y_test_all[:, i], y_pred[:, i], zero_division=0)
            support = y_test_all[:, i].sum()
            
            print(f"{appliance:<25} {precision:<10.3f} {recall:<10.3f} {f1:<10.3f} {support:<10.0f}")
            
            appliance_metrics[appliance] = {
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'support': support
            }
            f1_scores.append(f1)
    
    macro_f1 = np.mean(f1_scores) if f1_scores else 0
    print(f"\nOverall Macro F1-Score: {macro_f1:.4f}")
    
    # 2. Combination analysis
    print(f"\n{'='*70}")
    print("2. APPLIANCE COMBINATION ANALYSIS")
    print(f"{'='*70}")
    
    # Convert to combinations
    pred_combinations = []
    true_combinations = []
    
    for i in range(len(y_pred)):
        # Predicted combination
        pred_active = np.where(y_pred[i] == 1)[0]
        if len(pred_active) == 0:
            pred_comb = "No_Appliance"
        else:
            pred_comb = "+".join(sorted([appliance_names[idx] for idx in pred_active]))
        pred_combinations.append(pred_comb)
        
        # True combination
        true_active = np.where(y_test_all[i] == 1)[0]
        if len(true_active) == 0:
            true_comb = "No_Appliance"
        else:
            true_comb = "+".join(sorted([appliance_names[idx] for idx in true_active]))
        true_combinations.append(true_comb)
    
    # Count combinations
    pred_combo_counts = Counter(pred_combinations)
    true_combo_counts = Counter(true_combinations)
    
    print(f"\nMost Common Combinations in Test Data:")
    print("-" * 100)
    print(f"{'Combination':<80} {'True Count':<12} {'Pred Count':<12} {'Accuracy':<10}")
    print("-" * 100)
    
    # Show top 20 most common true combinations
    for combo, true_count in true_combo_counts.most_common(20):
        pred_count = pred_combo_counts.get(combo, 0)
        
        # Calculate accuracy for this combination
        correct = sum(1 for i in range(len(true_combinations)) 
                     if true_combinations[i] == combo and pred_combinations[i] == combo)
        accuracy = correct / true_count if true_count > 0 else 0
        
        print(f"{combo:<80} {true_count:<12} {pred_count:<12} {accuracy:<10.3f}")
    
    return y_pred, y_pred_probs, macro_f1, appliance_metrics, true_combo_counts, pred_combo_counts


test_client = 'Client_2'
X_test = clients_data[test_client]['X']
y_test = clients_data[test_client]['y']

print(f"\n{'='*70}")
print(f"EVALUATING FEDERATED MODEL ON {test_client}")
print(f"{'='*70}")
print(f"Test samples: {len(X_test)}")

In [1]:
# Evaluate federated model
print("\nFEDERATED MODEL EVALUATION:")
y_pred, y_pred_probs, federated_f1, federated_metrics, true_combs, pred_combs = evaluate_combination_performance(
    model=federated_model,
    X_test=X_test,
    y_test=y_test,
    appliance_names=appliance_names,
    batch_size=BATCH_SIZE,
    use_vae=USE_VAE
)


# Save combination analysis
combinations_data = []
for combo, count in true_combs.most_common(100):
    combinations_data.append({
        'Combination': combo,
        'True_Count': count,
        'Predicted_Count': pred_combs.get(combo, 0),
        'Accuracy': sum(1 for i in range(len(y_test)) 
                       if true_combs.most_common(1)[0][0] == combo and pred_combs.most_common(1)[0][0] == combo) / count if count > 0 else 0
    })

combinations_df = pd.DataFrame(combinations_data)
combinations_df.to_csv('appliance_combinations_results.csv', index=False)
print("Combination analysis saved to 'appliance_combinations_results.csv'")

Using device: cuda
Model architecture defined

LOADING YOUR PRETRAINED HYBRID MODEL
Pretrained Model Info:
    Architecture: HybridNILM
    Appliances: 15
    Window Size: 1000
    Use VAE: True
Error loading pretrained model: Error(s) in loading state_dict for HybridNILM:
	size mismatch for classifier.0.weight: copying a param with shape torch.Size([256, 960]) from checkpoint, the shape in current model is torch.Size([128, 960]).
	size mismatch for classifier.0.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for classifier.1.weight: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for classifier.1.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for classifier.1.running_mean: copying a param with shape torch.Size([256]) from checkpoint, the shap